In [11]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import *
import numpy as np

# Dataset

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/eduardofc/data/main/diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,Pedigree,Age,Diabetes
0,10,129,76,28,122,35.9,0.280,39,0
1,4,84,90,23,56,39.5,0.159,25,0
2,0,84,82,31,125,38.2,0.233,23,0
3,9,134,74,33,60,25.9,0.460,81,0
4,0,124,56,13,105,21.8,0.452,21,0


In [3]:
df.groupby('Diabetes').size()

Diabetes
0    529
1     21
dtype: int64

# Solucion clásica

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

models = []
models.append(("LR", LogisticRegression()))
models.append(("DTC", DecisionTreeClassifier()))
models.append(("NB", GaussianNB()))
models.append(("KNN", KNeighborsClassifier()))
models.append(("LDA", LinearDiscriminantAnalysis()))
models.append(("RF", RandomForestClassifier()))
models.append(("SVM", SVC()))

In [22]:
X = df.drop(columns=['Diabetes'])
y = df.Diabetes

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, stratify=y)

print('y_test')
print('Número de 0s:', np.count_nonzero(y_test==0))
print('Número de 1s:', np.count_nonzero(y_test==1))


y_test
Número de 0s: 106
Número de 1s: 4


In [25]:
for name, model in models:
    
    model.fit(X_train, y_train)
    acc = model.score(X_test, y_test)
    rec = recall_score(y_true=y_test, y_pred=y_hat)
    pre = precision_score(y_true=y_test, y_pred=y_hat)
    
    print(f"{name}: acc={acc:.2f} \t rec={rec:.2f} \t pre={pre:.2f}")

LR: acc=0.96 	 rec=0.00 	 pre=0.00
DTC: acc=0.95 	 rec=0.00 	 pre=0.00
NB: acc=0.95 	 rec=0.00 	 pre=0.00
KNN: acc=0.96 	 rec=0.00 	 pre=0.00
LDA: acc=0.97 	 rec=0.00 	 pre=0.00
RF: acc=0.96 	 rec=0.00 	 pre=0.00
SVM: acc=0.96 	 rec=0.00 	 pre=0.00


In [26]:
# y_hat = model.predict(X_test)
# confusion_matrix(y_true=y_test, y_pred=y_hat)

# Strategia 1: StratifiedKFolds

In [29]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

for name, model in models:
    
    grid_model = GridSearchCV(
        model, 
        cv=StratifiedKFold(n_splits=10, shuffle=True),
        param_grid={}
    )
    
    grid_model.fit(X_train, y_train)
    y_hat = grid_model.predict(X_test) 
    
    acc = accuracy_score(y_true=y_test, y_pred=y_hat)
    rec = recall_score(y_true=y_test, y_pred=y_hat)
    pre = precision_score(y_true=y_test, y_pred=y_hat)
    
    print(f"{name}: acc={acc:.2f} \t rec={rec:.2f} \t pre={pre:.2f}")

LR: acc=0.96 	 rec=0.00 	 pre=0.00
DTC: acc=0.95 	 rec=0.50 	 pre=0.40
NB: acc=0.95 	 rec=0.00 	 pre=0.00
KNN: acc=0.96 	 rec=0.00 	 pre=0.00
LDA: acc=0.97 	 rec=0.25 	 pre=1.00
RF: acc=0.96 	 rec=0.00 	 pre=0.00
SVM: acc=0.96 	 rec=0.00 	 pre=0.00


# Strategia 2: ROS

In [31]:
X = df.drop(columns=['Diabetes'])
y = df.Diabetes

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, stratify=y)

In [33]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=99)
X_resampled, y_resampled = ros.fit_resample(X_train, y_train)

print('y_train')
print('Número de 0s:', np.count_nonzero(y_train==0))
print('Número de 1s:', np.count_nonzero(y_train==1))

print('y_resampled')
print('Número de 0s:', np.count_nonzero(y_resampled==0))
print('Número de 1s:', np.count_nonzero(y_resampled==1))

y_train
Número de 0s: 423
Número de 1s: 17
y_resampled
Número de 0s: 423
Número de 1s: 423


In [34]:
for name, model in models:
    
    model.fit(X_resampled, y_resampled)
    y_hat = model.predict(X_test) 
    
    acc = accuracy_score(y_true=y_test, y_pred=y_hat)
    rec = recall_score(y_true=y_test, y_pred=y_hat)
    pre = precision_score(y_true=y_test, y_pred=y_hat)
    
    print(f"{name}: acc={acc:.2f} \t rec={rec:.2f} \t pre={pre:.2f}")

LR: acc=0.77 	 rec=0.50 	 pre=0.08
DTC: acc=0.92 	 rec=0.25 	 pre=0.14
NB: acc=0.80 	 rec=0.50 	 pre=0.09
KNN: acc=0.86 	 rec=0.00 	 pre=0.00
LDA: acc=0.86 	 rec=0.75 	 pre=0.18
RF: acc=0.95 	 rec=0.00 	 pre=0.00
SVM: acc=0.78 	 rec=0.50 	 pre=0.08


# EStrategia 3: RUS

In [35]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=99)
X_resampled, y_resampled = rus.fit_resample(X_train, y_train)

print('y_train')
print('Número de 0s:', np.count_nonzero(y_train==0))
print('Número de 1s:', np.count_nonzero(y_train==1))

print('y_resampled')
print('Número de 0s:', np.count_nonzero(y_resampled==0))
print('Número de 1s:', np.count_nonzero(y_resampled==1))

y_train
Número de 0s: 423
Número de 1s: 17
y_resampled
Número de 0s: 17
Número de 1s: 17


In [36]:
for name, model in models:
    
    model.fit(X_resampled, y_resampled)
    y_hat = model.predict(X_test) 
    
    acc = accuracy_score(y_true=y_test, y_pred=y_hat)
    rec = recall_score(y_true=y_test, y_pred=y_hat)
    pre = precision_score(y_true=y_test, y_pred=y_hat)
    
    print(f"{name}: acc={acc:.2f} \t rec={rec:.2f} \t pre={pre:.2f}")

LR: acc=0.64 	 rec=0.75 	 pre=0.07
DTC: acc=0.70 	 rec=0.25 	 pre=0.03
NB: acc=0.64 	 rec=1.00 	 pre=0.09
KNN: acc=0.70 	 rec=1.00 	 pre=0.11
LDA: acc=0.80 	 rec=0.75 	 pre=0.12
RF: acc=0.71 	 rec=0.75 	 pre=0.09
SVM: acc=0.71 	 rec=0.50 	 pre=0.06


# Estrategia 4: SMOTE

In [38]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=99)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

print('y_train')
print('Número de 0s:', np.count_nonzero(y_train==0))
print('Número de 1s:', np.count_nonzero(y_train==1))

print('y_resampled')
print('Número de 0s:', np.count_nonzero(y_resampled==0))
print('Número de 1s:', np.count_nonzero(y_resampled==1))

y_train
Número de 0s: 423
Número de 1s: 17
y_resampled
Número de 0s: 423
Número de 1s: 423


In [39]:
for name, model in models:
    
    model.fit(X_resampled, y_resampled)
    y_hat = model.predict(X_test) 
    
    acc = accuracy_score(y_true=y_test, y_pred=y_hat)
    rec = recall_score(y_true=y_test, y_pred=y_hat)
    pre = precision_score(y_true=y_test, y_pred=y_hat)
    
    print(f"{name}: acc={acc:.2f} \t rec={rec:.2f} \t pre={pre:.2f}")

LR: acc=0.73 	 rec=0.50 	 pre=0.07
DTC: acc=0.91 	 rec=0.50 	 pre=0.20
NB: acc=0.74 	 rec=0.50 	 pre=0.07
KNN: acc=0.80 	 rec=0.25 	 pre=0.05
LDA: acc=0.85 	 rec=0.50 	 pre=0.12
RF: acc=0.90 	 rec=0.25 	 pre=0.11
SVM: acc=0.78 	 rec=0.75 	 pre=0.12


# Estrategia 5: Algoritmos balanceados

In [44]:
from imblearn.ensemble import BalancedRandomForestClassifier

model = BalancedRandomForestClassifier(
    sampling_strategy='all',
    replacement=True,
    max_depth=10,
    bootstrap=False,
    random_state=99
)

model.fit(X_train, y_train)
y_hat = model.predict(X_test) 
    
acc = accuracy_score(y_true=y_test, y_pred=y_hat)
rec = recall_score(y_true=y_test, y_pred=y_hat)
pre = precision_score(y_true=y_test, y_pred=y_hat)

print(f"acc={acc:.2f} \t rec={rec:.2f} \t pre={pre:.2f}")

acc=0.86 	 rec=0.50 	 pre=0.13


# Estrategia 6: Class_weight

In [59]:
model = RandomForestClassifier(
    n_estimators=3,
    max_depth=5,
    random_state=67,
    class_weight={0:0.5, 1:20}
)

model.fit(X_train, y_train)
y_hat = model.predict(X_test) 
    
acc = accuracy_score(y_true=y_test, y_pred=y_hat)
rec = recall_score(y_true=y_test, y_pred=y_hat)
pre = precision_score(y_true=y_test, y_pred=y_hat)

print(f"acc={acc:.2f} \t rec={rec:.2f} \t pre={pre:.2f}")

acc=0.82 	 rec=0.50 	 pre=0.10
